## After sync

1. Confirm `$WORKSPACE_BUCKET/scripts/` and `$WORKSPACE_BUCKET/notebooks/` look fresh.
2. **Refresh WDL (AoU):** Workflows → `FlareByPopulation` → import/replace from  
   `gs://…/wdl/flare/wdl/FlareByPopulation.wdl` (same bucket as `WORKSPACE_BUCKET`).  
   Automatic Methods-repo Create on `allofus-drc-wgs-LR-prodData` is expected to **403**.
3. Confirm `flare_lai_exp` matches `flare/configs/lai_exp.tsv`.
4. To launch incomplete rows: `SUBMIT_FLARE = True` and re-run sync (or set `SUBMIT_ENTITY_IDS`).
5. Score finished rows with `flare_02_lai_exp_compare.ipynb` (Part 8).


## Config


In [ ]:
import os
from pathlib import Path

REPO_URL = os.environ.get("AOU_LR_REPO_URL", "https://github.com/kvg/aou-lr-phase-2.git")
REF = os.environ.get("AOU_LR_REF", "main")
CLONE_DIR = Path(os.environ.get("AOU_LR_REPO_DIR", str(Path.cwd() / "aou-lr-phase-2")))
# Keep notebooks outside the disposable git clone.
NOTEBOOK_DEST = Path(os.environ.get("AOU_LR_NOTEBOOK_DEST", str(Path.cwd() / "notebooks")))

GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN") or ""

UPSERT_TABLES = ["flare_lai_exp"]

# AoU billing-project Methods namespace is usually not writable (HTTP 403 Create).
# Leave empty; WDLs are staged to GCS for manual import. To auto-snapshot, set
# METHOD_NAMESPACE to an Agora namespace you own and uncomment UPDATE_METHODS.
UPDATE_METHODS = []
# UPDATE_METHODS = [{"wdl": "flare/wdl/FlareByPopulation.wdl", "name": "FlareByPopulation"}]
METHOD_NAMESPACE = os.environ.get("TERRA_METHOD_NAMESPACE", "")
BUMP_CONFIGS = []  # only useful after a successful method snapshot
# BUMP_CONFIGS = [{"config_name": "FlareByPopulation", "method_name": "FlareByPopulation"}]

SUBMIT_FLARE = os.environ.get("AOU_LR_SUBMIT_FLARE", "").lower() in {"1", "true", "yes"}
SUBMIT_ENTITY_IDS = None  # e.g. ["pin_gen8_afr_amr"]
SUBMIT_ONLY_INCOMPLETE = True
SUBMIT_CONFIG_NAME = "FlareByPopulation"

DRY_RUN = os.environ.get("AOU_LR_SYNC_DRY_RUN", "").lower() in {"1", "true", "yes"}

print("WORKSPACE_BUCKET:", os.environ.get("WORKSPACE_BUCKET", ""))
print("CLONE_DIR:", CLONE_DIR)
print("NOTEBOOK_DEST:", NOTEBOOK_DEST)
print("REF:", REF)
print("UPDATE_METHODS:", UPDATE_METHODS)
print("BUMP_CONFIGS:", BUMP_CONFIGS)
print("SUBMIT_FLARE:", SUBMIT_FLARE, "ids:", SUBMIT_ENTITY_IDS)
print("DRY_RUN:", DRY_RUN)


## Bootstrap

Clone/pull enough of the repo to import `terra_sync_repo` if it is not already on this VM.


In [ ]:
import subprocess
import sys

def _run(cmd, **kw):
    print("+", " ".join(map(str, cmd)) if isinstance(cmd, list) else cmd)
    return subprocess.run(cmd, check=True, text=True, **kw)

token = GITHUB_TOKEN
url = REPO_URL
auth_url = (
    f"https://x-access-token:{token}@" + url.split("https://", 1)[1]
    if token and url.startswith("https://")
    else url
)

CLONE_DIR.parent.mkdir(parents=True, exist_ok=True)
if CLONE_DIR.exists() and any(CLONE_DIR.iterdir()) and not (CLONE_DIR / ".git").is_dir():
    raise SystemExit(f"{CLONE_DIR} exists and is not a git checkout")

if (CLONE_DIR / ".git").is_dir():
    _run(["git", "remote", "set-url", "origin", auth_url], cwd=str(CLONE_DIR))
    _run(["git", "fetch", "--tags", "--force", "origin"], cwd=str(CLONE_DIR))
    # Disposable mirror: discard Jupyter autosaves inside the clone.
    _run(["git", "reset", "--hard", f"origin/{REF}"], cwd=str(CLONE_DIR))
    _run(["git", "clean", "-fd"], cwd=str(CLONE_DIR))
    _run(["git", "checkout", "-B", REF, f"origin/{REF}"], cwd=str(CLONE_DIR))
else:
    try:
        _run(["git", "clone", "--branch", REF, "--single-branch", auth_url, str(CLONE_DIR)])
    except subprocess.CalledProcessError:
        _run(["git", "clone", auth_url, str(CLONE_DIR)])
        _run(["git", "checkout", REF], cwd=str(CLONE_DIR))
if token:
    _run(["git", "remote", "set-url", "origin", REPO_URL], cwd=str(CLONE_DIR))

scripts = CLONE_DIR / "scripts"
assert (scripts / "terra_sync_repo.py").is_file(), scripts
if str(scripts) not in sys.path:
    sys.path.insert(0, str(scripts))
print("import path:", scripts)
print("HEAD:", _run(["git", "rev-parse", "--short", "HEAD"], cwd=str(CLONE_DIR), capture_output=True).stdout.strip())



## Sync + optional submit


In [ ]:
import os
from pathlib import Path

REPO_URL = os.environ.get("AOU_LR_REPO_URL", "https://github.com/kvg/aou-lr-phase-2.git")
REF = os.environ.get("AOU_LR_REF", "main")
CLONE_DIR = Path(os.environ.get("AOU_LR_REPO_DIR", str(Path.cwd() / "aou-lr-phase-2")))
# Keep notebooks outside the disposable git clone.
NOTEBOOK_DEST = Path(os.environ.get("AOU_LR_NOTEBOOK_DEST", str(Path.cwd() / "notebooks")))

GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN") or ""

UPSERT_TABLES = ["flare_lai_exp"]

# AoU billing-project Methods namespace is usually not writable (HTTP 403 Create).
# Leave empty; WDLs are staged to GCS for manual import. To auto-snapshot, set
# METHOD_NAMESPACE to an Agora namespace you own and uncomment UPDATE_METHODS.
UPDATE_METHODS = []
# UPDATE_METHODS = [{"wdl": "flare/wdl/FlareByPopulation.wdl", "name": "FlareByPopulation"}]
METHOD_NAMESPACE = os.environ.get("TERRA_METHOD_NAMESPACE", "")
BUMP_CONFIGS = []  # only useful after a successful method snapshot
# BUMP_CONFIGS = [{"config_name": "FlareByPopulation", "method_name": "FlareByPopulation"}]

SUBMIT_FLARE = os.environ.get("AOU_LR_SUBMIT_FLARE", "").lower() in {"1", "true", "yes"}
SUBMIT_ENTITY_IDS = None  # e.g. ["pin_gen8_afr_amr"]
SUBMIT_ONLY_INCOMPLETE = True
SUBMIT_CONFIG_NAME = "FlareByPopulation"

DRY_RUN = os.environ.get("AOU_LR_SYNC_DRY_RUN", "").lower() in {"1", "true", "yes"}

print("WORKSPACE_BUCKET:", os.environ.get("WORKSPACE_BUCKET", ""))
print("CLONE_DIR:", CLONE_DIR)
print("NOTEBOOK_DEST:", NOTEBOOK_DEST)
print("REF:", REF)
print("UPDATE_METHODS:", UPDATE_METHODS)
print("BUMP_CONFIGS:", BUMP_CONFIGS)
print("SUBMIT_FLARE:", SUBMIT_FLARE, "ids:", SUBMIT_ENTITY_IDS)
print("DRY_RUN:", DRY_RUN)


## After sync

1. Confirm `$WORKSPACE_BUCKET/scripts/` and `$WORKSPACE_BUCKET/notebooks/` look fresh.
2. **Refresh WDL (AoU):** Workflows → `FlareByPopulation` → import/replace from  
   `gs://…/wdl/flare/wdl/FlareByPopulation.wdl` (same bucket as `WORKSPACE_BUCKET`).  
   Automatic Methods-repo Create on `allofus-drc-wgs-LR-prodData` is expected to **403**.
3. Confirm `flare_lai_exp` matches `flare/configs/lai_exp.tsv`.
4. To launch incomplete rows: `SUBMIT_FLARE = True` and re-run sync (or set `SUBMIT_ENTITY_IDS`).
5. Score finished rows with `flare_02_lai_exp_compare.ipynb` (Part 8).
